# 07 — Collaboration networks and leadership (Book 2, Chapter 7)

In [1]:
import itertools, numpy as np, pandas as pd, networkx as nx
from pathlib import Path
ROOT = Path("..").resolve() if Path("../config.yml").exists() else Path(".").resolve()
FIN, LOGS = ROOT/"data/final", ROOT/"logs"
pc = pd.read_parquet(FIN/"paper_country.parquet"); papers = pd.read_parquet(FIN/"papers.parquet")
# PubMed stored only the FIRST author's affiliation before late 2013 (all authors from 2014).
# Before 2014 the network would show only first-author countries, so we start in 2015.
pc = pc[pc["year"] >= 2015]
per = pc.groupby("pmid")["country"].apply(lambda s: sorted(set(s)))
multi = per[per.map(len) > 1]
print(f"papers 2015–2026 with a known country: {len(per):,}; with two or more countries: {len(multi):,} ({len(multi)/len(per):.1%})")

papers 2015–2026 with a known country: 5,074; with two or more countries: 2,674 (52.7%)


## Fractional edge weights
A paper with k countries contributes 1/(k−1) to each of its country pairs, so that a
30-country consortium paper does not outweigh 30 bilateral papers.

In [2]:
full, frac = {}, {}
for cs in multi:
    w = 1 / (len(cs) - 1)
    for a, b in itertools.combinations(cs, 2):
        full[(a, b)] = full.get((a, b), 0) + 1; frac[(a, b)] = frac.get((a, b), 0) + w
edges = pd.DataFrame([(a, b, full[(a, b)], round(frac[(a, b)], 1)) for a, b in full], columns=["a","b","papers","fractional"])
print("Top pairs by full count\n", edges.nlargest(10, "papers").to_string(index=False))
print("\nTop pairs by fractional weight\n", edges.nlargest(10, "fractional").to_string(index=False))
edges.to_csv(FIN/"country_edges_2015_2026.csv", index=False)

Top pairs by full count
   a   b  papers  fractional
GBR USA     344       146.8
AUS USA     165        77.8
CHE USA     163        63.8
KEN USA     130        56.6
DEU USA     127        44.9
FRA USA     127        48.1
GBR KEN     126        52.6
NLD USA     123        51.2
MLI USA     112        71.1
AUS GBR     109        37.1

Top pairs by fractional weight
   a   b  papers  fractional
GBR USA     344       146.8
AUS USA     165        77.8
MLI USA     112        71.1
CHE USA     163        63.8
KEN USA     130        56.6
GBR KEN     126        52.6
NLD USA     123        51.2
FRA USA     127        48.1
GBR NLD     103        47.6
CAN USA      82        47.3


In [3]:
G = nx.Graph(); [G.add_edge(r.a, r.b, weight=r.fractional) for r in edges.itertuples()]
deg = pd.Series(dict(G.degree())); strength = pd.Series(dict(G.degree(weight="weight")))
btw = pd.Series(nx.betweenness_centrality(G, weight=None))
cent = pd.DataFrame({"partners": deg, "strength": strength.round(1), "betweenness": btw.round(3)}).sort_values("strength", ascending=False)
print("countries:", G.number_of_nodes(), "| links:", G.number_of_edges(), "| density:", round(nx.density(G), 3))
cent.head(12)

countries: 149 | links: 2509 | density: 0.228


,partners,strength,betweenness
USA,128,1362.7,0.157
GBR,122,782.4,0.108
AUS,89,344.9,0.029
DEU,85,309.0,0.029
FRA,89,288.4,0.049
NLD,74,285.2,0.012
CHE,93,282.6,0.038
KEN,84,250.8,0.012
GHA,83,219.3,0.011
IND,80,177.2,0.018


## Leadership: who holds the first and last author positions?
Among papers with at least one sub-Saharan African country, and a known country for that position.

In [4]:
p = papers[(papers["year"] >= 2015) & papers["any_ssa"].fillna(False).astype(bool)].copy()
p["period"] = pd.cut(p["year"], [2014, 2019, 2022, 2026], labels=["2015–19","2020–22","2023–26"])
lead = p.groupby("period", observed=True).apply(lambda g: pd.Series({
    "papers_with_SSA": len(g),
    "first_known": int(g.known_first.sum()), "SSA_first_%": round(100 * g.loc[g.known_first.fillna(False).astype(bool), "ssa_first"].astype(float).mean(), 1),
    "last_known": int(g.known_last.sum()), "SSA_last_%": round(100 * g.loc[g.known_last.fillna(False).astype(bool), "ssa_last"].astype(float).mean(), 1)}), include_groups=False)
lead.to_csv(LOGS/"leadership_ssa.csv"); lead

,papers_with_SSA,first_known,SSA_first_%,last_known,SSA_last_%
period,,,,,
2015–19,413.0,400.0,53.2,375.0,39.2
2020–22,377.0,374.0,54.8,351.0,46.2
2023–26,658.0,645.0,63.1,616.0,52.6


## OpenAlex: leadership over a longer period, and institutions
OpenAlex holds countries for all author positions in most older papers too (notebook 03), so it
lets us look before 2014. We first check that it agrees with PubMed where both are complete.

In [5]:
oe_file = FIN / "openalex_enrich.parquet"
if oe_file.exists():
    oe = pd.read_parquet(oe_file); oe = oe[oe.openalex_id.notna()].copy()
    region = pd.read_parquet(FIN / "paper_country_openalex.parquet").drop_duplicates("country").set_index("country")["region"].to_dict()
    ssa = lambda cs: any(region.get(c) == "Sub-Saharan Africa" for c in cs)
    oe["any_ssa_oa"] = oe.oa_countries.map(lambda cs: ssa(list(cs)))
    oe = oe[oe.any_ssa_oa & (oe.year >= 2000)]
    oe["period"] = pd.cut(oe.year, [1999, 2009, 2014, 2019, 2022, 2026], labels=["2000–09", "2010–14", "2015–19", "2020–22", "2023–26"])
    lead_oa = oe.groupby("period", observed=True).apply(lambda g: pd.Series({
        "papers_with_SSA": len(g),
        "SSA_first_%": round(100 * g[g.oa_first.map(len) > 0].oa_first.map(lambda c: ssa(list(c))).mean(), 1),
        "SSA_last_%": round(100 * g[g.oa_last.map(len) > 0].oa_last.map(lambda c: ssa(list(c))).mean(), 1),
        "n_last_known": int((g.oa_last.map(len) > 0).sum())}), include_groups=False)
    lead_oa.to_csv(LOGS / "leadership_ssa_openalex.csv"); print(lead_oa.to_string())

    # Institution network, 2015–2026, fractional weights
    inst = oe_all = pd.read_parquet(oe_file)
    inst = inst[(inst.year >= 2015) & inst.institutions.map(lambda x: x is not None and len(x) > 0)]
    names, ifull, ifrac = {}, {}, {}
    for lst in inst.institutions:
        ids = sorted({i[0] for i in lst}); names.update({i[0]: i[1] for i in lst})
        if len(ids) < 2: continue
        w = 1 / (len(ids) - 1)
        for a, b in itertools.combinations(ids, 2):
            ifull[(a, b)] = ifull.get((a, b), 0) + 1; ifrac[(a, b)] = ifrac.get((a, b), 0) + w
    IG = nx.Graph(); [IG.add_edge(a, b, weight=ifrac[(a, b)]) for (a, b) in ifrac]
    npapers = inst.institutions.explode().dropna().map(lambda t: t[0]).value_counts()
    ib = pd.Series(nx.betweenness_centrality(IG, k=500, seed=2026))
    itab = pd.DataFrame({"institution": [names[i] for i in npapers.index[:15]], "papers": npapers.values[:15],
                         "partners": [IG.degree(i) if i in IG else 0 for i in npapers.index[:15]],
                         "betweenness": [round(ib.get(i, 0), 3) for i in npapers.index[:15]]})
    print("institutions:", IG.number_of_nodes(), "| links:", IG.number_of_edges()); print(itab.to_string(index=False))
    top_b = ib.sort_values(ascending=False).head(10)
    print("Highest betweenness:\n", pd.Series({names[i]: round(v, 3) for i, v in top_b.items()}).to_string())

         papers_with_SSA  SSA_first_%  SSA_last_%  n_last_known
period                                                         
2000–09            361.0         62.0        45.0         302.0
2010–14            320.0         57.3        47.2         284.0
2015–19            401.0         52.4        39.3         384.0
2020–22            347.0         56.8        47.3         328.0
2023–26            604.0         62.1        52.5         566.0


institutions: 3589 | links: 38252
                                          institution  papers  partners  betweenness
National Institute of Allergy and Infectious Diseases     394       505        0.039
                                 University of Oxford     390       617        0.066
                        National Institutes of Health     374       512        0.043
         London School of Hygiene & Tropical Medicine     248       549        0.064
                                     Jenner Institute     213       306        0.011
                          The University of Melbourne     203       340        0.028
                     Kenya Medical Research Institute     185       381        0.018
                             Johns Hopkins University     173       378        0.052
                                   Mahidol University     146       325        0.026
                    Radboud University Medical Center     144       281        0.008
               Walter Reed Army